# Model Training & Evaluation

Train and evaluate multiple sentiment classifiers:
- Logistic Regression (baseline)
- Support Vector Machine (SVM)
- Naive Bayes

Metrics: macro F1, precision, recall per class.

In [ ]:
# Cell 1: Imports
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import pickle
import logging
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from data_preprocessing import preprocess_dataframe, create_tfidf_features
from model_training import save_model

logging.basicConfig(level=logging.INFO)
print("Imports successful")

In [ ]:
# Cell 2: Load train/test sets
train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Train class distribution:\n{train['Sentiment'].value_counts()}\n")
print(f"Test class distribution:\n{test['Sentiment'].value_counts()}")

In [ ]:
# Cell 3: Preprocess (suppress debug output)
import warnings
logging.getLogger('data_preprocessing').setLevel(logging.WARNING)

print("Preprocessing train set...")
train_proc = preprocess_dataframe(train, text_column='Comment')
print("Preprocessing test set...")
test_proc = preprocess_dataframe(test, text_column='Comment')
print(f"Preprocessed train: {train_proc.shape}, test: {test_proc.shape}")

In [ ]:
# Additional sanity checks after preprocessing
# Ensure preprocess_dataframe returned a valid DataFrame with expected columns
assert train_proc is not None, "train_proc is None — preprocess_dataframe returned None"
assert test_proc is not None, "test_proc is None — preprocess_dataframe returned None"
print("train_proc type:", type(train_proc))
print("train_proc columns:", list(train_proc.columns)[:50])
assert 'Sentiment' in train_proc.columns, "'Sentiment' column missing in train_proc"
assert 'cleaned_text' in train_proc.columns, "'cleaned_text' missing — preprocess didn't add it"

# Show sample cleaned_text values
print("Sample cleaned_text:")
print(train_proc['cleaned_text'].head(3).to_list())

In [ ]:
# Cell 4: Create TF-IDF features
print("Creating TF-IDF features...")
# increase max_features to include more tokens (was too small)
vectorizer, X_train, X_test = create_tfidf_features(train_proc, test_proc, max_features=10000)

# Ensure TF-IDF returned valid matrices
assert X_train is not None and X_test is not None, "TF-IDF returned None for X_train or X_test"

# Normalize to scipy CSR matrices to ensure .shape/.nnz are available and satisfy type checkers
from scipy import sparse
import numpy as np

def to_csr(matrix):
    try:
        return sparse.csr_matrix(matrix)
    except Exception:
        return sparse.csr_matrix(np.asarray(matrix))

X_train = to_csr(X_train)
X_test = to_csr(X_test)

# Use labels from the processed DataFrames (keeps alignment) and convert to numpy arrays
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Fit encoder on sentiment strings (ensure mapping to original class names)
le.fit(pd.concat([train_proc['Sentiment'], test_proc['Sentiment']]).astype(str))

# Raw label arrays (strings) and encoded integer arrays for training
y_train_raw = train_proc['Sentiment'].astype(str).to_numpy()
y_test_raw = test_proc['Sentiment'].astype(str).to_numpy()

y_train = le.transform(y_train_raw)
y_test = le.transform(y_test_raw)

# Sanity checks for alignment
assert X_train.shape[0] == y_train.shape[0], f"X_train rows {X_train.shape[0]} != y_train length {y_train.shape[0]}"
assert X_test.shape[0] == y_test.shape[0], f"X_test rows {X_test.shape[0]} != y_test length {y_test.shape[0]}"

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Feature count: {len(vectorizer.get_feature_names_out())}")

In [ ]:
# Reload data_preprocessing module to pick up recent code changes
import importlib
import data_preprocessing
importlib.reload(data_preprocessing)

# Recreate TF-IDF features using updated function
vectorizer, X_train, X_test = data_preprocessing.create_tfidf_features(train_proc, test_proc, max_features=10000)
print('X_train shape:', X_train.shape)
print('Feature count:', len(vectorizer.get_feature_names_out()))

# --- Supervised feature selection (SelectKBest using chi2) ---
from sklearn.feature_selection import SelectKBest, chi2
import joblib, os

# Choose k conservatively: up to 5000 or number of features
k = min(5000, X_train.shape[1])
if k < 1:
    k = 1
print(f'Applying SelectKBest(chi2) with k={k}')

try:
    selector = SelectKBest(chi2, k=k)
    X_train_sel = selector.fit_transform(X_train, y_train)
    X_test_sel = selector.transform(X_test)
    print('X_train_sel shape:', X_train_sel.shape)
    print('X_test_sel shape:', X_test_sel.shape)

    # Persist selector for inference
    os.makedirs('../models', exist_ok=True)
    joblib.dump(selector, '../models/selector.joblib')
    print('Saved selector to ../models/selector.joblib')

    # Replace X matrices for downstream training cells
    X_train = X_train_sel
    X_test = X_test_sel
except Exception as e:
    print('Feature selection skipped due to error:', e)


In [ ]:
# Diagnostic: show TF-IDF vocabulary and top tokens
print('Number of features in vectorizer:', len(vectorizer.get_feature_names_out()))
print(vectorizer.get_feature_names_out())

# Show top 20 idf values (lowest idf => most common)
idf = vectorizer.idf_
indices = idf.argsort()[:20]
print('Top tokens (most common):')
for i in indices:
    print(vectorizer.get_feature_names_out()[i], idf[i])

In [ ]:
# Diagnostic: cleaned_text statistics
train_ct = train_proc['cleaned_text']
non_empty = train_ct.str.strip().replace('', pd.NA).dropna()
print('Rows with non-empty cleaned_text:', non_empty.shape[0], ' / ', train_proc.shape[0])

# token counts distribution (sample)
token_counts = non_empty.str.split().map(len)
print('Avg tokens per non-empty doc:', token_counts.mean())
print('Top tokens frequency (sample 20):')
from collections import Counter
c = Counter()
for t in non_empty.head(1000):
    c.update(t.split())
print(c.most_common(20))

In [ ]:
# Check presence of specific token in vectorizer
for token in ['খুশি', 'ভালো', 'খুব']:
    print(token, 'in vocab?', token in vectorizer.vocabulary_)

In [ ]:
# Show some sample cleaned_text rows
print(train_proc['cleaned_text'].head(20).to_list())

In [ ]:
# Quick test: fit a fresh vectorizer on a sample of documents
from sklearn.feature_extraction.text import TfidfVectorizer as TF2
sample_docs = train_proc['cleaned_text'].head(5000).astype(str).tolist()
vec2 = TF2(max_features=10000, tokenizer=lambda x: x.split(), preprocessor=lambda x: x, lowercase=False, min_df=1)
X2 = vec2.fit_transform(sample_docs)
print('vec2 feature count:', len(vec2.get_feature_names_out()))
print('Is খুশি in vec2 vocab?', 'খুশি' in vec2.vocabulary_)
print('Some features:', vec2.get_feature_names_out()[:30])

In [ ]:
# Cell 5: Train models
models = {}

# Verify shapes before training
print(f"Training data rows: {X_train.shape[0]}, labels: {y_train.shape[0]}")

print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial', solver='lbfgs')
lr.fit(X_train, y_train)
models['LogisticRegression'] = lr

print("Training Linear SVM...")
svm = LinearSVC(max_iter=2000, random_state=42, dual=False)
svm.fit(X_train, y_train)
models['LinearSVM'] = svm

print("Training Multinomial Naive Bayes...")
nb = MultinomialNB()
nb.fit(X_train, y_train)
models['NaiveBayes'] = nb

print("All models trained!")

In [ ]:
# Cell 6: Evaluate models
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)

    results[name] = {'accuracy': acc, 'f1_macro': f1_macro, 'precision_macro': prec_macro, 'recall_macro': rec_macro}
    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  F1 (macro): {f1_macro:.4f}")
    print(f"  Precision (macro): {prec_macro:.4f}")
    print(f"  Recall (macro):    {rec_macro:.4f}")

    # Detailed report
    try:
        print("  Classification report:")
        print(classification_report(y_test, y_pred, target_names=le.classes_))
    except Exception:
        pass

# Convert to DataFrame for comparison
results_df = pd.DataFrame(results).T
print("\n=== Model Comparison ===")
print(results_df)

In [ ]:
# Cell 7: Select best model and save artifacts
best_model_name = results_df['f1_macro'].idxmax()
best_model = models[best_model_name]
best_f1 = results_df.loc[best_model_name, 'f1_macro']

print(f"\nBest model: {best_model_name} (F1={best_f1:.4f})")

import os
import joblib
os.makedirs('../models', exist_ok=True)

# Save using joblib for sklearn objects
joblib.dump(best_model, '../models/best_model.joblib')
joblib.dump(vectorizer, '../models/vectorizer.joblib')
joblib.dump(le, '../models/label_encoder.joblib')

print("Saved best_model.joblib, vectorizer.joblib and label_encoder.joblib to models/")

summary = {
    'best_model': best_model_name,
    'best_f1': float(best_f1),
    'results': results
}
joblib.dump(summary, '../models/training_summary.joblib')
print("Training complete!")